In [ ]:
# Loading dataset

In [5]:
%run data_loader_panel.py



Start of data summary


File loaded successfully.
rd_panel_data.csv loaded successfully as df_panel_data.

df_panel_data structure:

|   # | Column                                                |   Non‑Null Count | Dtype   |
|-----|-------------------------------------------------------|------------------|---------|
|   0 | Ticker                                                |          169,193 | str     |
|   1 | SimFinId                                              |          169,194 | int64   |
|   2 | Currency                                              |          169,194 | str     |
|   3 | Fiscal Year                                           |          169,194 | int64   |
|   4 | Fiscal Period                                         |          169,194 | str     |
|   5 | Report Date                                           |          169,194 | str     |
|   6 | Publish Date                                          |          169,194 | str     |
|   7 | Restated Date       

In [ ]:
# Identify firms whose last observed fiscal year is the year they left the panel,
# export a summary and per-year lists for external bankruptcy labeling.

df = df_panel_data.copy()
df['entity_id'] = df['Ticker'].fillna(df['SimFinId'].astype(str))
df['Fiscal Year'] = df['Fiscal Year'].astype(int)

last_obs = (df.groupby('entity_id', as_index=False)
              .agg(last_year=('Fiscal Year','max'),
                   Ticker=('Ticker','first'),
                   SimFinId=('SimFinId','first'),
                   CompanyName=('Company Name','first')))

# Save master table of last observation years
out_base = file_path.replace('.csv','') + '_dropouts_last_year'
last_obs.to_csv(out_base + '.csv', index=False)

# Save per-year sheets to an Excel file (fallback to CSV if ExcelWriter not available)
try:
    with pd.ExcelWriter(out_base + '.xlsx') as writer:
        for year, grp in last_obs.groupby('last_year'):
            grp[['entity_id','Ticker','SimFinId','CompanyName']].to_excel(writer, sheet_name=str(int(year)), index=False)
except Exception:
    # already saved CSV fallback
    pass

# Print counts of dropouts by year
print(last_obs['last_year'].value_counts().sort_index())

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
